# 4 Datenobjekte 

Gültig für Digitalisate aus der Sosa, bzw. Objekte, die auf lokalen Laufwerken liegen. 
Achtung! Dieses Notebook kann in der Ausführung sehr lange dauern, da die Dateien riesig sind. 
Voraussetzung: mit VPN verbunden / im unilu-Netzwerk, mit Laufwerk G verbunden.

Aufgrund der geringen Menge und der diversen Metadatenquellen werden diese Zipkapseln von Hand vom Laufwerk G auf die Workbench verschoben. Sie werden alle manuell entzippt. 

Die Zipkapseln sind alle nach folgender Struktur benannt:

    DOI/ID _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiele:

    000190118_20150320T000256_master_ver1.zip
    10_7891_e-manuscripta-108732.zip

Die Objekte liegen auf G:\ZHB-Sosa_Digital\digital. Die vollständigen Pfade auf G sind in der Eingabedatei ergänzt und ist in die Infojson unter 'additional' abgelegt. 
Der Dateipfad auf der Workbench wird nach dem DOI benannt. 

Dieses Script erstellt in 'object' die Unterordner mit dem Identifier, die Files werden ohne weitere Zwischenverarbeitung oder Prüfung von G hierhin kopiert. 
Das Entzippen kann auf der Workbench erfolgen (manuell). 



In [1]:
import os
import config
import json
from datetime import datetime
import zipfile
import shutil
from pathlib import Path

localdrive = config.user_root
org_id = config.organisation_id
file_name = f'{config.inventory_file}'
objects_path = f'{localdrive}/{config.collection_id}/{config.object_path}'
counter = 0

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        counter += 1
        
        # get path to G drive:
        g_path = value["additional"]
        #print(f"Origin path: {g_path}")
        
        # create new object folder name (AIP path). The full path is needed here for copying the files.
        #foldername = value["signature"][(len(org_id)+1):]
        foldername = value["references"][-1]
        aip_path = f"{objects_path}/{foldername}"
         
        print("Destination path:",aip_path)
                
        # prepare object folder: make a directory for each object
        Path(f'{aip_path}').mkdir(parents=True, exist_ok=True)
        
        
        filenames = []
        # Iterate directory, check if current file_path is a file
        try:
            for file_path in os.listdir(g_path):
                print("Origin files:", file_path)
                if os.path.isfile(os.path.join(g_path, file_path)):
                    filenames.append(file_path)
                else:
                    print("---------------- not a file!-----------------------")
        except FileNotFoundError:
            print(f"The directory {g_path} does not exist")
        except PermissionError:
            print(f"Permission denied to access the directory {g_path}")
        except OSError as e:
            print(f"An OS error occurred: {e}")
        
        for file in filenames:   
            aip_file = Path(aip_path+'/'+file)
            #print("File to be copied:",aip_path+'/'+file)
            if aip_file.exists():
                # path exists
                print("*** Path exists, file already copied")
            else:
                # copy file:
                print("Copying file:", file)
                print("Time started copying:",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))
                shutil.copy(g_path+'/'+file, aip_path+'/'+file) 
            
                print("File copied successfully.")
                
        #debugging:        
        if counter == 1:
            break
        

            
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

Destination path: C:/Users/HeimK/switchdrive/jupyter/dlza/sosa_emanus/objects/10_7891_e-manuscripta-108732
Origin files: 10_7891_e-manuscripta-108732
---------------- not a file!-----------------------
Origin files: 10_7891_e-manuscripta-108732.zip
Copying file: 10_7891_e-manuscripta-108732.zip
Time started copying: 2024-01-05 17:14:01
File copied successfully.
Finished at  2024-01-05 17:14:03
